In [12]:
# =============================================================================
# CELL 1: ALL CONFIGURATION, ASSUMPTIONS, BASELINES, INPUTS
# =============================================================================

# --- Granularity: 'q' = quarterly, 'm' = monthly, 'w' = weekly ---
granularity = 'm'

# --- Date Range (inclusive) ---
START_DATE = '2025-10-07'
END_DATE = None  # None = auto-detect from today's date

# --- Query Control ---
run_every_query = True  # True = run SQL queries; False = use cached pickles

# --- Date Column per Granularity ---
# Q/M use book_date; W uses app_date (application_received_dtm)
DATE_COL_MAP = {'q': 'book_date', 'm': 'book_date', 'w': 'app_date'}

# --- LOBs to Process ---
LOBS = ['STE']

# --- Rollup Groups ---
ROLLUP_GROUPS = {}  # Single LOB; no rollups needed

# --- Baselines (STE-specific) ---
BASELINES = {
    'STE': {'ltv': 1.94, 'new_recovery_unadjusted': 0.557, 'apr': 0.25},
}

# --- Model Parameters ---
MODEL_PARAMS = {
    'mmi_standard_increase': 1.03,
    'expected_years_on_book': 2,
    'impound_probability': 0.15,
    'mean_unit_loss': 0.5,
    'unit_loss_to_model_score': 0.02,
    'skip_rate': 0.75,
    'kmx_loss_scale': 0.067,
}

# --- Impact Score Bounds (floor, ceiling) ---
IMPACT_BOUNDS = {
    'gross_loss_impact': (-20, 10),
    'recovery_impact': (-10, 20),
    'ltv_impact': (-15, 15),
    'apr_impact': (-10, 10),
}

# --- STE does NOT use DLA -- uniform pricing scalar ---
PRICING_SCALAR = 1.0

# --- Dual-path override: join SFS data at account level ---
USE_STE_METRICS = True

# --- Excluded Vintages ---
EXCLUDED_VINTAGES = {}

In [13]:
# =============================================================================
# CELL 2: IMPORTS AND DERIVED CONFIGURATION
# =============================================================================
import pandas as pd
import numpy as np
import pyodbc
import pickle
import warnings
import time
import datetime as dt
import re
import os
import openpyxl
from tqdm.notebook import tqdm

tqdm.pandas()
pd.set_option('display.max_columns', 100)
pd.set_option('display.min_rows', 100)

# --- Derived values (do not modify) ---
date_col = DATE_COL_MAP[granularity]
start_date = pd.Timestamp(START_DATE)
end_date = pd.Timestamp(END_DATE) if END_DATE else pd.Timestamp.today().normalize()

PERIOD_FREQ_MAP = {'q': 'Q', 'm': 'M', 'w': 'W-SAT'}
period_freq = PERIOD_FREQ_MAP[granularity]

start_period = start_date.to_period(period_freq)
end_period = end_date.to_period(period_freq)

min_date_sql = f"'{START_DATE}'"

print(f"Granularity: {granularity}")
print(f"Date column: {date_col}")
print(f"Period range: {start_period} to {end_period}")
print(f"SQL min_date: {min_date_sql}")

Granularity: m
Date column: book_date
Period range: 2025-10 to 2026-08
SQL min_date: '2025-10-07'


In [14]:
# =============================================================================
# CELL 3: UTILITY FUNCTIONS
# =============================================================================

def run_sql(filename, sub_list=None, connection=None, filename_is_query=False):
    if sub_list is None:
        sub_list = []
    if filename_is_query:
        query = filename
    else:
        with open(filename, 'r') as file:
            query = file.read()
    for text, var in sub_list:
        query = query.replace(text, var)
    if connection is None:
        with pyodbc.connect("DSN=Redshift_prod_new") as conn:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=query, con=conn)
            warnings.filterwarnings("default", category=UserWarning)
            return df
    else:
        warnings.filterwarnings("ignore", category=UserWarning)
        df = pd.read_sql_query(sql=query, con=connection)
        warnings.filterwarnings("default", category=UserWarning)
        return df


def store_pickle(data, filename):
    if isinstance(data, str):
        data, filename = filename, data
    with open(filename, 'wb') as file:
        pickle.dump(data, file)


def get_pickle(filename):
    with open(filename, 'rb') as file:
        return pickle.load(file)


def cached_sql(filename, pickle_name, sub_list=None, connection=None, force_refresh=False):
    """Fetch from SQL and cache to pickle. Reuse cache unless force_refresh=True
    or the pickle file is missing."""
    if force_refresh or not os.path.exists(pickle_name):
        df = run_sql(filename, sub_list=sub_list, connection=connection)
        if df.empty:
            print(f"  WARNING: query '{filename}' returned 0 rows")
        store_pickle(df, pickle_name)
        return df
    df = get_pickle(pickle_name)
    if df.empty:
        print(f"  WARNING: cached '{pickle_name}' contains 0 rows")
    return df


def weighted_average_and_sum(group, metrics):
    """Per-metric population-aware weighted average: each metric's denominator
    only includes amt_financed from accounts where that metric is not NaN."""
    if isinstance(metrics, str):
        valid = group[metrics].notna()
        if valid.any():
            weighted_avg = (group.loc[valid, metrics] * group.loc[valid, 'amt_financed']).sum() / group.loc[valid, 'amt_financed'].sum()
        else:
            weighted_avg = np.nan
        return pd.Series({metrics: weighted_avg, 'amt_financed': group.amt_financed.sum()})
    result_dict = {'amt_financed': group.amt_financed.sum()}
    for metric in metrics:
        valid = group[metric].notna()
        if valid.any():
            result_dict[metric] = (
                (group.loc[valid, metric] * group.loc[valid, 'amt_financed']).sum()
                / group.loc[valid, 'amt_financed'].sum()
            )
        else:
            result_dict[metric] = np.nan
    return pd.Series(result_dict)


def assign_period(df, col_name, freq):
    """Assign a pd.Period column from a date column."""
    dt_series = pd.to_datetime(df[col_name])
    return dt_series.dt.to_period(freq)


def format_vintage(period_series):
    """Convert pd.Period series to formatted vintage strings."""
    if period_series.empty:
        return period_series.astype(str)
    freq = period_series.iloc[0].freqstr
    if freq == 'Q-DEC':
        return period_series.dt.year.astype(str) + ' Q' + period_series.dt.quarter.astype(str)
    elif freq == 'M':
        return period_series.dt.year.astype(str) + ' M' + period_series.dt.month.astype(str).str.zfill(2)
    return period_series.astype(str)

In [15]:
# =============================================================================
# CELL 4: ULA MULTIPLIER FUNCTIONS (nonKMX path only)
# =============================================================================

def get_ula_multiplier_nonkmx(ula_df, leave_out='None'):
    ula_df['loss_multiplier'] = 1.0

    if leave_out != 'Previous ACA chargeoff':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.prev_co_flag

    if leave_out != 'Small amount financed':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.small_amt_financed_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Zero cash down':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.zero_cash_down_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'High mileage vehicle':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_mileage_vehicle_flag

    if leave_out != 'High PTI':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.high_pti_flag * (1 - ula_df.pricing_change_flag)

    if leave_out != 'Car make':
        ula_df.loss_multiplier *= (1 + 0.1 * ula_df.car_make_penalty_flag
                                   - 0.1 * ula_df.car_make_benefit_flag
                                   - 0.1 * ula_df.pricing_change_flag * ula_df.car_make_benefit_flag)

    if leave_out != 'Theft risk':
        ula_df.loss_multiplier *= (0.987 + 0.099 * ula_df.theft_risk_flag * (1 - ula_df.pricing_change_flag)
                                   + 0.013 * ula_df.pricing_change_flag)

    if leave_out != 'MCY high model score, low mileage':
        ula_df.loss_multiplier *= 1 - 0.2 * ula_df.mcy_low_mileage_flag

    if leave_out != 'Weekday/weekend decision':
        ula_df.loss_multiplier *= 1 - 0.05 * ula_df.weekend_flag + 0.02 * ula_df.weekday_flag

    if leave_out != 'Student Loans':
        ula_df.loss_multiplier *= 1 + (0.1 * ula_df.student_loan_flag
            - np.minimum(np.maximum((ula_df.cd_model_score - (130 - 2 * ula_df.ent_flag)) * 0.006, 0), 0.03)
            * (1 - ula_df.student_loan_flag)) * ula_df.student_loans_cutoff_date

    if leave_out != 'Low PTI':
        ula_df.loss_multiplier *= 1 + (-0.03 * np.minimum(ula_df.cd_model_score, 135) + 3.9) * ula_df.low_pti_flag

    if leave_out != 'Secured credit (Chime, etc.)':
        ula_df.loss_multiplier *= 0.966 + 0.273 * ula_df.nonkmx_chime_flag

    if leave_out != 'Employment type':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.seasonal_employment_flag - 0.1 * ula_df.waiter_employment_flag

    if leave_out != 'Authorized tradelines':
        ula_df.loss_multiplier *= 1 + 0.1 * ula_df.nonkmx_auth_tradelines_flag

    if leave_out != 'Fraud':
        ula_df.loss_multiplier *= 1 + (ula_df.fraud_adjustment - 1)

    if leave_out != 'Driver flag':
        ula_df.loss_multiplier *= 1 + 0.15 * ula_df.driver_flag

    if leave_out != 'Clip':
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.8, 2)

    if leave_out != 'Dealer Level (Non-KMX)':
        ula_df.loss_multiplier *= ula_df.pricing_scalar
        ula_df.loss_multiplier = np.clip(ula_df.loss_multiplier, 0.7, 1.4)

    return ula_df

In [16]:
# =============================================================================
# CELL 5: DATA FETCH (SQL + PICKLE)
# =============================================================================

os.makedirs('cache', exist_ok=True)
force = run_every_query

need_conn = force or not all(
    os.path.exists(p) for p in (
        '../../cache/ste_indiv_ula_v1.pkl',
        '../../cache/ste_indiv_recovery_v1.pkl',
        '../../cache/ste_indiv_weekly_v1.pkl',
    )
)

if need_conn:
    conn = pyodbc.connect("DSN=Redshift_prod_new")

    with open('../queries/ste_ragu_temptables.txt', 'r') as f:
        conn.execute(f.read().strip())
    print('Temp tables created')

    ula_df_total = cached_sql('../queries/ste_ragu_ula.txt', '../../cache/ste_indiv_ula_v1.pkl',
                              connection=conn, force_refresh=force)
    print(f'ULA ready: {len(ula_df_total):,} records')

    new_recovery = cached_sql('../queries/ste_ragu_recovery.txt', '../../cache/ste_indiv_recovery_v1.pkl',
                              connection=conn, force_refresh=force)
    print(f'Recovery ready: {len(new_recovery):,} records')

    ste_weekly_raw = cached_sql('../queries/ste_ragu_weekly.txt', '../../cache/ste_indiv_weekly_v1.pkl',
                                connection=conn, force_refresh=force)
    print(f'STE weekly metrics ready: {len(ste_weekly_raw):,} records')

    conn.close()
else:
    ula_df_total = get_pickle('../../cache/ste_indiv_ula_v1.pkl')
    new_recovery = get_pickle('../../cache/ste_indiv_recovery_v1.pkl')
    ste_weekly_raw = get_pickle('../../cache/ste_indiv_weekly_v1.pkl')
    print('All data loaded from cache')

print(f"ULA records: {len(ula_df_total):,}")
print(f"STE weekly records: {len(ste_weekly_raw):,}")

Temp tables created
ULA ready: 81,635 records
Recovery ready: 22,041 records
STE weekly metrics ready: 22,140 records
ULA records: 81,635
STE weekly records: 22,140


In [17]:
# =============================================================================
# CELL 6: PERIOD ASSIGNMENT + DATE FILTERING + FLAG CREATION
# =============================================================================

# Filter out Core LOB if present
ula_df_total = ula_df_total[ula_df_total.lob != 'Core']

# Relabel lob from 'STG' to 'STE'
ula_df_total['lob'] = 'STE'

# Ensure date columns are proper types
ula_df_total['app_date'] = pd.to_datetime(ula_df_total['app_date'])
ula_df_total['book_date'] = pd.to_datetime(ula_df_total['book_date'])
new_recovery['app_date'] = pd.to_datetime(new_recovery['app_date'])
new_recovery['book_date'] = pd.to_datetime(new_recovery['book_date'])

# Assign period columns
for df in [ula_df_total, new_recovery]:
    df['quarter'] = pd.to_datetime(df[date_col]).dt.to_period('Q')
    df['month'] = pd.to_datetime(df[date_col]).dt.to_period('M')
    df['week'] = pd.to_datetime(df[date_col]).dt.to_period('W-SAT')

    period_key = {'q': 'quarter', 'm': 'month', 'w': 'week'}
    df['period'] = df[period_key[granularity]]

# Filter to configured date range
for df in [ula_df_total, new_recovery]:
    mask = (df['period'] >= start_period) & (df['period'] <= end_period)
    df.drop(df[~mask].index, inplace=True)

# Convert week columns to str
ula_df_total['book_week'] = ula_df_total['book_week'].astype(str)

# String version of date_col for flag comparisons
ula_df_total[f'{date_col}_str'] = ula_df_total[date_col].astype(str)

# STE-specific caps
ula_df_total['bbltv'] = ula_df_total.amt_financed / ula_df_total.bbvalue.replace(0, np.nan)
ula_df_total = ula_df_total[
    (ula_df_total.bbltv <= 10.0) |
    (ula_df_total.bbvalue.isna()) |
    (ula_df_total.bbvalue == 0)
]
ula_df_total = ula_df_total[ula_df_total.pti <= 0.6]
ula_df_total = ula_df_total[ula_df_total.total_income <= 200000]

print(f"Periods in data: {ula_df_total['period'].nunique()}")
print(f"Period range: {ula_df_total['period'].min()} to {ula_df_total['period'].max()}")
print(f"ULA after caps: {len(ula_df_total):,}")

# =============================================================================
# FLAG CREATION
# =============================================================================

date_col_str = f'{date_col}_str'

ula_df_total['mtn_3_1_flag'] = ula_df_total.mtn_model == 'MTN3.1'
ula_df_total['vehicle_age'] = np.maximum(ula_df_total[date_col_str].str[:4].astype(int) - ula_df_total.model_year, 1/365)
ula_df_total.tradein_value = ula_df_total.tradein_value.fillna(0)
ula_df_total.make = ula_df_total.make.str.upper().str[:3]
ula_df_total.lob_or_bucket = np.select(
    [ula_df_total.lob_or_bucket.isna() & ula_df_total.lob.isin(['Core', 'FRN']),
     ula_df_total.lob_or_bucket.isna() & ~ula_df_total.lob.isin(['Core', 'FRN'])],
    ['D', 'C'], default=ula_df_total.lob_or_bucket)
ula_df_total['continuous_vehicle_age'] = (ula_df_total[date_col_str].str[:4].astype(int)
    + ula_df_total[date_col_str].str[5:7].astype(int) / 12
    - (ula_df_total.model_year - 0.25) - 1)

# No DLA: uniform pricing scalar
ula_df_total['pricing_scalar'] = PRICING_SCALAR

# Driver Flag
warnings.filterwarnings("ignore", category=UserWarning)
ula_df_total.job_company = ula_df_total.job_company.fillna('')
ula_df_total['driver_flag'] = np.where(
    ula_df_total.job_company.str.contains('(LYFT)|(UBER)|(GRUB ?HUB)|(DOOR ?DASH)|(GO ?PUFF)|(POST ?MATE)|(INSTA ?CART)|(DOMINO)|(PAPA J)|(PIZZA)|(JIMMY ?JOHN)|(SELF)'),
    1, 0)
warnings.filterwarnings("default", category=UserWarning)

# NA Handling
ula_df_total = ula_df_total.dropna(subset=['lob'])
ula_df_total.cash_down = ula_df_total.cash_down.fillna(0)
ula_df_total.specialty_dealer = ula_df_total.specialty_dealer.fillna('not specialty')
ula_df_total.prev_co_count = ula_df_total.prev_co_count.fillna(0)

# NonKMX Flags (STE uses PTI threshold = 0.2)
ula_df_total['ent_fld_flag'] = False
ula_df_total['small_amt_financed_flag'] = (ula_df_total.bbvalue > 0) & (ula_df_total.bbvalue < 5000) & (ula_df_total.amt_financed < 4500)
ula_df_total['zero_cash_down_flag'] = (ula_df_total.cash_down <= 250) & (ula_df_total.tradein_value < 3000)
ula_df_total['high_mileage_vehicle_flag'] = (ula_df_total.mileage >= 100000) & (ula_df_total.cd_model_score < 130)
ula_df_total['high_pti_flag'] = (ula_df_total.pti > 0.2) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_penalty_flag'] = ula_df_total.make.isin({'CAD', 'CHR', 'BMW', 'BUI', 'SUB'}) & (ula_df_total.cd_model_score < 130) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['car_make_benefit_flag'] = (ula_df_total.cd_model_score >= 133) & ula_df_total.make.isin({'HON', 'TOY', 'LEX'}) & (ula_df_total.specialty_dealer != 'Ally')
ula_df_total['theft_risk_flag'] = ula_df_total.make.isin({'KIA', 'HYU'}) & (ula_df_total.specialty_dealer != 'Ally') & (ula_df_total.model_year >= 2015) & (ula_df_total.model_year <= 2021) & (ula_df_total[date_col_str] >= '2022-07-01') & (ula_df_total[date_col_str] < '2025-01-01')
ula_df_total['mcy_low_mileage_flag'] = False
ula_df_total['weekend_flag'] = ula_df_total.day_of_week.isin([0, 6])
ula_df_total['weekday_flag'] = ula_df_total.day_of_week.isin(range(1, 6))
ula_df_total['secured_credit_flag'] = ula_df_total.secured_credit_card
ula_df_total['chime_flag'] = ula_df_total.chime_indicator
ula_df_total['nonkmx_chime_flag'] = ula_df_total.nonkmx_chime_indicator
ula_df_total['seasonal_employment_flag'] = ula_df_total['seasonal_employment_flag'] == 1 if 'seasonal_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'seasonal')
ula_df_total['waiter_employment_flag'] = ula_df_total['waiter_employment_flag'] == 1 if 'waiter_employment_flag' in ula_df_total.columns else (ula_df_total.get('employment', '') == 'waiter')
ula_df_total['nonkmx_auth_tradelines_flag'] = ula_df_total.pct_auth_tradelines > 0.2
ula_df_total['prev_co_flag'] = ula_df_total.prev_co_count > 0
ula_df_total['null_fico_w_vantage_flag'] = ((ula_df_total.fico_score < 300) | (ula_df_total.fico_score > 850)) & (ula_df_total.vantage_score >= 300) & (ula_df_total.vantage_score <= 850)
ula_df_total['pricing_change_flag'] = ula_df_total[date_col_str] >= '2024-10-01'
ula_df_total['ent_flag'] = False
ula_df_total['student_loans_cutoff_date'] = ula_df_total[date_col_str] >= '2023-05-01'
ula_df_total['low_pti_flag'] = (ula_df_total.pti <= 0.05) & (ula_df_total.cd_model_score >= 130) & (ula_df_total.cb_flag)
ula_df_total['student_loan_flag'] = 0

# Deduplicate driver flags
combined_driver_flag_df = ula_df_total.groupby('account_number').driver_flag.max().reset_index()
ula_df_total = ula_df_total.drop(columns='driver_flag').merge(combined_driver_flag_df, on='account_number')
ula_df_total = ula_df_total.drop(columns='job_company').drop_duplicates()

# Vintage strings
ula_df_total['vintage'] = format_vintage(ula_df_total['period'])
new_recovery['vintage'] = format_vintage(new_recovery['period'])

print(f"ULA after refinement: {len(ula_df_total):,}")

# =============================================================================
# SFS DATA PREPARATION (for account-level override)
# =============================================================================

ste_filtered = ste_weekly_raw.copy()
ste_date_col = 'application_received_dtm' if granularity == 'w' else 'book_date'
ste_filtered[ste_date_col] = pd.to_datetime(ste_filtered[ste_date_col])
ste_filtered['period'] = ste_filtered[ste_date_col].dt.to_period(period_freq)
ste_filtered = ste_filtered[(ste_filtered.period >= start_period) & (ste_filtered.period <= end_period)]
ste_filtered['vintage'] = format_vintage(ste_filtered['period'])

# Apply same STE caps to SFS data
ste_filtered = ste_filtered[ste_filtered['con_pti_back'] <= 0.6]
ste_filtered = ste_filtered[ste_filtered['total_income'] <= 200000]
ste_filtered['bbltv'] = ste_filtered['con_amount_financed_back'] / ste_filtered['bb_value'].replace(0, np.nan)
ste_filtered = ste_filtered[
    (ste_filtered['bbltv'] <= 10.0) |
    (ste_filtered['bb_value'].isna()) |
    (ste_filtered['bb_value'] == 0)
]

# Prepare SFS columns for account-level join
sfs_override_cols = ste_filtered[['account_number', 'con_risk_model_score', 'bbltv', 'con_apr']].copy()
sfs_override_cols = sfs_override_cols.rename(columns={
    'con_risk_model_score': 'sfs_model_score',
    'bbltv': 'sfs_ltv',
    'con_apr': 'sfs_apr',
})
sfs_override_cols = sfs_override_cols.drop_duplicates(subset='account_number', keep='first')

print(f"SFS override records available: {len(sfs_override_cols):,}")
print(f"STE filtered total: {len(ste_filtered):,}")

Periods in data: 11
Period range: 2025-10 to 2026-08
ULA after caps: 79,251
ULA after refinement: 20,741
SFS override records available: 20,641
STE filtered total: 20,737


In [18]:
# =============================================================================
# CELL 7: ACCOUNT-LEVEL RAGU COMPUTATION WITH SFS OVERRIDE + JENSEN'S
# =============================================================================

mean_unit_loss = MODEL_PARAMS['mean_unit_loss']
unit_loss_to_model_score = MODEL_PARAMS['unit_loss_to_model_score']

# --- Apply ULA multipliers (nonKMX path only for STE) ---
ula_all = get_ula_multiplier_nonkmx(ula_df_total.copy())

# --- Merge recovery multiplier ---
nr = new_recovery[['account_number', 'new_recovery_multiplier']].drop_duplicates(
    subset='account_number', keep='first')
acct_df = ula_all.merge(
    nr.rename(columns={'new_recovery_multiplier': 'recovery_multiplier'}),
    on='account_number', how='left'
).drop_duplicates(subset='account_number', keep='first')

# --- SFS Override: join at account level ---
acct_df = acct_df.merge(sfs_override_cols, on='account_number', how='left')

# Use SFS values where available, fall back to ULA
has_sfs = acct_df['sfs_model_score'].notna()
print(f"Total accounts: {len(acct_df):,}")
print(f"  with SFS match: {has_sfs.sum():,} ({has_sfs.mean()*100:.1f}%)")
print(f"  without SFS (fallback to ULA): {(~has_sfs).sum():,}")

# model_score: SFS override or ULA fallback, capped at 160
acct_df['model_score'] = np.where(has_sfs, acct_df['sfs_model_score'], acct_df['cd_model_score'])
acct_df['model_score'] = acct_df['model_score'].clip(upper=160)

# LTV: SFS override or ULA-computed fallback
has_bb = acct_df['bbvalue'].notna() & (acct_df['bbvalue'] > 0)
ula_ltv = np.where(has_bb, acct_df.amt_financed / acct_df.bbvalue, np.nan)
has_sfs_ltv = acct_df['sfs_ltv'].notna()
acct_df['ltv'] = np.where(has_sfs_ltv, acct_df['sfs_ltv'], ula_ltv)

# APR: SFS override or ULA fallback
has_sfs_apr = acct_df['sfs_apr'].notna()
acct_df['apr'] = np.where(has_sfs_apr, acct_df['sfs_apr'], acct_df['apr'])

print(f"  with bbvalue > 0: {has_bb.sum():,}")
print(f"  with recovery: {acct_df['recovery_multiplier'].notna().sum():,}")

# --- Per-account RAGU computation ---
acct_df['unit_loss_score'] = (
    acct_df.model_score
    + (1 - acct_df.loss_multiplier) * mean_unit_loss / unit_loss_to_model_score
)

baseline_config = BASELINES['STE']
baseline_recovery = baseline_config['new_recovery_unadjusted']
baseline_ltv = baseline_config['ltv']
baseline_apr = baseline_config['apr']

acct_df['baseline_recovery'] = baseline_recovery
acct_df['baseline_ltv'] = baseline_ltv
acct_df['baseline_apr'] = baseline_apr
acct_df['ltv_mult'] = 17
acct_df['apr_mult'] = 0.7

acct_df['baselined_recovery'] = acct_df.recovery_multiplier / baseline_recovery

# RAGU decomposition components (unclipped)
acct_df['gross_loss_impact'] = acct_df.unit_loss_score - acct_df.model_score
acct_df['recovery_impact'] = (
    acct_df.unit_loss_score * mean_unit_loss
    * acct_df.recovery_multiplier * (acct_df.baselined_recovery - 1)
)
acct_df['ltv_impact'] = (baseline_ltv / acct_df.ltv - 1) * acct_df.ltv_mult
acct_df['apr_impact'] = (baseline_apr - acct_df.apr) / 0.01 * acct_df.apr_mult

# Preserve unclipped for _agg score
acct_df['gross_loss_impact_unclipped'] = acct_df['gross_loss_impact'].copy()
acct_df['apr_impact_unclipped'] = acct_df['apr_impact'].copy()

# Apply impact bounds
for col, (floor, ceil) in IMPACT_BOUNDS.items():
    acct_df[col] = acct_df[col].clip(lower=floor, upper=ceil)

acct_df['ragu_score'] = (
    acct_df.model_score
    + acct_df.gross_loss_impact
    + acct_df.recovery_impact
    + acct_df.ltv_impact
    + acct_df.apr_impact
)

# =============================================================================
# JENSEN'S CORRECTION
# =============================================================================

def jensen_group_stats(grp):
    M = MODEL_PARAMS['mean_unit_loss']

    scored = grp[grp['recovery_multiplier'].notna() & grp['ltv'].notna()]
    rec_stats = {}
    if len(scored) > 1:
        w = scored['amt_financed'].values
        R = scored['recovery_multiplier'].values
        ULS = scored['unit_loss_score'].values
        B = scored['baseline_recovery'].iloc[0]
        mu_R = np.average(R, weights=w)
        mu_ULS = np.average(ULS, weights=w)
        var_R = np.average((R - mu_R) ** 2, weights=w)
        cov_ULS_R = np.average((ULS - mu_ULS) * (R - mu_R), weights=w)
        mixed_3rd = np.average((ULS - mu_ULS) * (R - mu_R) ** 2, weights=w)
        delta_RI_taylor = M * (mu_ULS * var_R / B + cov_ULS_R * (2 * mu_R / B - 1) + mixed_3rd / B)
        unclipped_RI = ULS * M * R * (R / B - 1)
        agg_RI = np.average(unclipped_RI, weights=w)
        target_RI = M * mu_ULS * mu_R * (mu_R / B - 1)
        delta_RI_exact = agg_RI - target_RI
        rec_stats = {
            'mu_R': mu_R, 'mu_ULS': mu_ULS, 'var_R': var_R,
            'cov_ULS_R': cov_ULS_R, 'mixed_3rd': mixed_3rd,
            'delta_RI_taylor': delta_RI_taylor, 'delta_RI_exact': delta_RI_exact,
            'agg_RI': agg_RI, 'target_RI': target_RI, 'B_rec': B,
        }

    bb = grp[grp['ltv'].notna()]
    ltv_stats = {}
    if len(bb) > 1:
        w = bb['amt_financed'].values
        L = bb['ltv'].values
        B_ltv = bb['baseline_ltv'].iloc[0]
        lm = bb['ltv_mult'].iloc[0]
        mu_L = np.average(L, weights=w)
        unclipped_LI = (B_ltv / L - 1) * lm
        agg_LI = np.average(unclipped_LI, weights=w)
        target_LI = (B_ltv / mu_L - 1) * lm
        delta_LI = agg_LI - target_LI
        ltv_stats = {
            'mu_L': mu_L, 'delta_LI': delta_LI, 'agg_LI': agg_LI,
            'target_LI': target_LI, 'B_ltv': B_ltv, 'ltv_mult_val': lm,
        }

    return {**rec_stats, **ltv_stats}

group_stats = {}
for (vintage, lob), grp in acct_df.groupby(['vintage', 'lob']):
    group_stats[(vintage, lob)] = jensen_group_stats(grp)

# --- Unclipped per-loan values (base for adjusted columns) ---
acct_df['_ri_unclipped'] = (
    acct_df['unit_loss_score'] * mean_unit_loss
    * acct_df['recovery_multiplier'] * (acct_df['baselined_recovery'] - 1)
)
acct_df['_li_unclipped'] = (acct_df['baseline_ltv'] / acct_df['ltv'] - 1) * acct_df['ltv_mult']

# --- Initialize adjusted columns from unclipped base ---
acct_df['recovery_impact_adj_prop'] = acct_df['_ri_unclipped'].copy()
acct_df['recovery_impact_adj_contrib'] = acct_df['_ri_unclipped'].copy()
acct_df['recovery_impact_adj_exact'] = acct_df['_ri_unclipped'].copy()
acct_df['ltv_impact_adj_prop'] = acct_df['_li_unclipped'].copy()
acct_df['ltv_impact_adj_contrib'] = acct_df['_li_unclipped'].copy()
acct_df['ltv_impact_adj_exact'] = acct_df['_li_unclipped'].copy()

for (vintage, lob), stats in group_stats.items():
    mask = (acct_df.vintage == vintage) & (acct_df.lob == lob)
    scored_mask = mask & acct_df['_ri_unclipped'].notna()
    bb_mask = mask & acct_df['ltv'].notna()

    # --- Recovery adjustments ---
    if 'agg_RI' in stats and stats['agg_RI'] != 0:
        M = MODEL_PARAMS['mean_unit_loss']
        B = stats['B_rec']

        scale_prop_ri = (stats['agg_RI'] - stats['delta_RI_taylor']) / stats['agg_RI']
        acct_df.loc[scored_mask, 'recovery_impact_adj_prop'] *= scale_prop_ri

        scale_exact_ri = stats['target_RI'] / stats['agg_RI']
        acct_df.loc[scored_mask, 'recovery_impact_adj_exact'] *= scale_exact_ri

        sub = acct_df.loc[scored_mask]
        contrib = M * (
            sub['unit_loss_score'] * (sub['recovery_multiplier'] - stats['mu_R']) ** 2 / B
            + (sub['unit_loss_score'] - stats['mu_ULS'])
              * (sub['recovery_multiplier'] - stats['mu_R'])
              * (2 * stats['mu_R'] / B - 1)
        )
        w = sub['amt_financed']
        avg_contrib = (contrib * w).sum() / w.sum()
        if avg_contrib != 0:
            share = contrib * stats['delta_RI_taylor'] / avg_contrib
            acct_df.loc[scored_mask, 'recovery_impact_adj_contrib'] -= share.values

    # --- LTV adjustments ---
    if 'agg_LI' in stats and stats['agg_LI'] != 0:
        scale_li = stats['target_LI'] / stats['agg_LI']
        acct_df.loc[bb_mask, 'ltv_impact_adj_prop'] *= scale_li
        acct_df.loc[bb_mask, 'ltv_impact_adj_exact'] *= scale_li

        sub = acct_df.loc[bb_mask]
        B_ltv = stats['B_ltv']
        lm = stats['ltv_mult_val']
        mu_L = stats['mu_L']
        contrib_ltv = (B_ltv / sub['ltv'] - B_ltv / mu_L) * lm
        w = sub['amt_financed']
        avg_contrib_ltv = (contrib_ltv * w).sum() / w.sum()
        if avg_contrib_ltv != 0:
            share_ltv = contrib_ltv * stats['delta_LI'] / avg_contrib_ltv
            acct_df.loc[bb_mask, 'ltv_impact_adj_contrib'] -= share_ltv.values

# --- RAGU score agg (uses adj_exact variants) ---
acct_df['ragu_score_agg'] = (
    acct_df.model_score
    + acct_df.gross_loss_impact_unclipped
    + acct_df.recovery_impact_adj_exact
    + acct_df.ltv_impact_adj_exact
    + acct_df.apr_impact_unclipped
)

print(f"Jensen's correction applied to {len(group_stats)} vintage-LOB groups")

# Select output columns
output_cols = [
    'account_number', 'lob', 'vintage', 'book_date', 'app_date', 'model_score',
    'gross_loss_impact',
    'recovery_impact', 'recovery_impact_adj_prop', 'recovery_impact_adj_contrib', 'recovery_impact_adj_exact',
    'ltv_impact', 'ltv_impact_adj_prop', 'ltv_impact_adj_contrib', 'ltv_impact_adj_exact',
    'apr_impact', 'ragu_score', 'ragu_score_agg', 'amt_financed',
    'ltv', 'apr', 'recovery_multiplier', 'unit_loss_score',
]
acct_df = acct_df[output_cols].copy()

print(f"\nAccount-level RAGU computed: {len(acct_df):,} accounts")
print(f"  with full RAGU (ltv + recovery): {acct_df['ragu_score'].notna().sum():,}")
print(f"Vintages: {acct_df.vintage.nunique()}")
display(acct_df.head(10))

Total accounts: 20,741
  with SFS match: 20,640 (99.5%)
  without SFS (fallback to ULA): 101
  with bbvalue > 0: 20,400
  with recovery: 19,622
Jensen's correction applied to 11 vintage-LOB groups

Account-level RAGU computed: 20,741 accounts
  with full RAGU (ltv + recovery): 19,417
Vintages: 11


,account_number,lob,vintage,book_date,app_date,model_score,gross_loss_impact,recovery_impact,recovery_impact_adj_prop,recovery_impact_adj_contrib,recovery_impact_adj_exact,ltv_impact,ltv_impact_adj_prop,ltv_impact_adj_contrib,ltv_impact_adj_exact,apr_impact,ragu_score,ragu_score_agg,amt_financed,ltv,apr,recovery_multiplier,unit_loss_score
0,90125158207,STE,2026 M01,2026-01-14,2026-01-03,122.0,-0.236750,-1.326434,-0.979016,-1.922171,-0.979016,3.854482,3.411178,4.432585,3.411178,-1.400000,122.891299,122.795411,28110.00,1.581435,0.2700,0.534287,121.763250
1,90125118651,STE,2025 M11,2025-11-13,2025-11-07,134.0,-5.836232,4.493973,3.317401,4.425937,3.317401,5.775675,5.071762,4.350490,5.071762,4.900001,143.333417,141.452932,34101.25,1.448036,0.1800,0.620002,128.163768
2,90125297562,STE,2026 M06,2026-06-25,2026-06-20,130.0,2.057500,-3.637317,-2.956557,-5.494636,-2.956557,1.929138,1.730423,6.169639,1.730423,-0.973000,129.376321,129.858367,25393.84,1.742287,0.2639,0.495015,132.057500
3,90125280684,STE,2026 M06,2026-06-02,2026-05-29,141.0,-5.646665,1.318245,1.071522,1.157136,1.071522,11.182112,10.030277,6.169639,10.030277,1.827000,149.680693,148.282134,20420.79,1.170246,0.2239,0.575841,135.353335
4,90125291269,STE,2026 M06,2026-06-16,2026-06-13,137.0,2.745775,-2.727026,-2.216636,-3.666144,-2.216636,3.171010,2.844374,6.169639,2.844374,-1.183000,139.006759,139.190513,20683.00,1.635020,0.2669,0.514770,139.745775
5,90125142444,STE,2025 M12,2025-12-22,2025-12-13,132.0,2.332810,0.446358,0.328308,0.426151,0.328308,4.720581,4.113811,3.959939,4.113811,3.080000,142.579748,141.854929,26154.02,1.518376,0.2060,0.563568,134.332810
6,90125265330,STE,2026 M05,2026-05-14,2026-05-02,132.0,-4.073135,-3.540982,-2.842916,-5.414958,-2.842916,2.169465,1.945818,6.423008,1.945818,-0.336000,126.219348,126.693767,17419.50,1.720444,0.2548,0.494664,127.926865
7,90125268594,STE,2026 M05,2026-05-19,2026-04-25,135.0,-3.543463,-2.768871,-2.223018,-4.021946,-2.223018,5.887976,5.280992,6.423008,5.280992,-1.525999,133.049643,132.988512,20533.27,1.440931,0.2718,0.511090,131.456537
8,90125208554,STE,2026 M03,2026-03-12,2026-02-27,132.0,-6.215366,-2.175424,-1.397682,-2.730871,-1.397682,5.198343,4.710467,5.567605,4.710467,0.000000,128.807553,129.097419,18942.63,1.485696,0.2500,0.519945,125.784634
9,90125135975,STE,2025 M12,2025-12-09,2025-12-04,137.0,1.105990,-3.772741,-2.774950,-4.434856,-2.774950,8.566926,7.465758,3.959939,7.465758,-1.385999,141.514175,141.410799,21509.88,1.289948,0.2698,0.495595,138.105990


In [19]:
# =============================================================================
# CELL 8: VINTAGE-LOB AGGREGATION
# =============================================================================

agg_metrics = [
    'model_score', 'gross_loss_impact',
    'recovery_impact', 'recovery_impact_adj_prop', 'recovery_impact_adj_contrib', 'recovery_impact_adj_exact',
    'ltv_impact', 'ltv_impact_adj_prop', 'ltv_impact_adj_contrib', 'ltv_impact_adj_exact',
    'apr_impact', 'ragu_score', 'ragu_score_agg', 'ltv', 'apr',
]


def population_aware_agg(grp):
    """Aggregate metrics using the correct non-NaN population for each metric's
    denominator."""
    result = {}

    full_pop = grp
    bb_pop = grp[grp['ltv'].notna()]
    scored_pop = grp[grp['recovery_impact_adj_exact'].notna() & grp['ltv'].notna()]

    full_metrics = ['model_score', 'gross_loss_impact']
    for m in full_metrics:
        sub = full_pop[full_pop[m].notna()]
        if len(sub) > 0:
            result[m] = (sub[m] * sub['amt_financed']).sum() / sub['amt_financed'].sum()
        else:
            result[m] = np.nan

    bb_metrics = [
        'ltv_impact', 'ltv_impact_adj_prop', 'ltv_impact_adj_contrib', 'ltv_impact_adj_exact',
        'apr_impact', 'ltv', 'apr',
    ]
    for m in bb_metrics:
        sub = bb_pop[bb_pop[m].notna()]
        if len(sub) > 0:
            result[m] = (sub[m] * sub['amt_financed']).sum() / sub['amt_financed'].sum()
        else:
            result[m] = np.nan

    scored_metrics = [
        'recovery_impact', 'recovery_impact_adj_prop', 'recovery_impact_adj_contrib', 'recovery_impact_adj_exact',
        'ragu_score', 'ragu_score_agg',
    ]
    for m in scored_metrics:
        sub = scored_pop[scored_pop[m].notna()]
        if len(sub) > 0:
            result[m] = (sub[m] * sub['amt_financed']).sum() / sub['amt_financed'].sum()
        else:
            result[m] = np.nan

    result['amt_financed'] = full_pop['amt_financed'].sum()
    return pd.Series(result)


# --- Aggregate account-level to vintage-LOB level ---
vintage_lob_df = acct_df.groupby(['vintage', 'lob']).apply(
    population_aware_agg, include_groups=False
).reset_index()

print(f"Vintage-LOB aggregation: {len(vintage_lob_df)} rows "
      f"({vintage_lob_df.vintage.nunique()} vintages x {vintage_lob_df.lob.nunique()} LOBs)")
print(f"Groups: {sorted(vintage_lob_df.lob.unique())}")

display(vintage_lob_df.sort_values(['lob', 'vintage']).head(20))

# --- Population breakdown ---
print("\n--- Population breakdown (first 5 vintage-LOB combos) ---")
for (vintage, lob), grp in list(acct_df.groupby(['vintage', 'lob']))[:5]:
    n_total = len(grp)
    n_bb = grp['ltv'].notna().sum()
    n_ragu = grp['ragu_score_agg'].notna().sum()
    af_total = grp['amt_financed'].sum()
    print(f"  {lob} {vintage}: {n_total:,} total | {n_bb:,} bb>0 | {n_ragu:,} scored | "
          f"AF total={af_total:,.0f}")

Vintage-LOB aggregation: 11 rows (11 vintages x 1 LOBs)
Groups: ['STE']


,vintage,lob,model_score,gross_loss_impact,ltv_impact,ltv_impact_adj_prop,ltv_impact_adj_contrib,ltv_impact_adj_exact,apr_impact,ltv,apr,recovery_impact,recovery_impact_adj_prop,recovery_impact_adj_contrib,recovery_impact_adj_exact,ragu_score,ragu_score_agg,amt_financed
0,2025 M10,STE,131.021335,-0.381944,5.167138,4.656834,4.656834,4.656834,0.897726,1.522845,0.237175,3.711513,2.978682,2.984370,2.978682,140.090855,138.846145,3.270728e+07
1,2025 M11,STE,131.365156,-0.391560,4.894763,4.350490,4.350490,4.350490,1.083953,1.544695,0.234515,2.848958,2.145102,2.149554,2.145102,139.653353,138.394426,4.813154e+07
2,2025 M12,STE,131.651392,-0.469095,4.507679,3.959939,3.959939,3.959939,1.073263,1.573478,0.234668,2.665815,1.983097,1.981740,1.983097,139.242640,138.000396,4.906867e+07
3,2026 M01,STE,132.248966,-0.375392,4.949619,4.432585,4.432585,4.432585,1.083966,1.538778,0.234515,2.770388,2.068785,2.069245,2.068785,140.714295,139.491130,4.206601e+07
4,2026 M02,STE,133.111325,-0.401917,5.496605,4.977842,4.977842,4.977842,1.181530,1.500602,0.233121,1.950252,1.274731,1.273518,1.274731,141.372814,140.169387,5.189566e+07
5,2026 M03,STE,134.396978,-0.355943,6.074084,5.567605,5.567605,5.567605,1.500677,1.461387,0.228562,1.718093,1.130682,1.130635,1.130682,143.259055,142.167582,1.078950e+08
6,2026 M04,STE,135.069052,-0.279018,6.338505,5.803167,5.803167,5.803167,1.665240,1.446290,0.226211,3.007122,2.297731,2.297512,2.297731,145.714137,144.468842,9.062796e+07
7,2026 M05,STE,136.238790,-0.393591,6.947647,6.423008,6.423008,6.423008,1.781983,1.408017,0.224543,4.213113,3.449309,3.449198,3.449309,148.585508,147.285660,7.272996e+07
8,2026 M06,STE,136.526672,-0.576008,6.682790,6.169639,6.169639,6.169639,1.882104,1.423415,0.223113,4.847027,4.014976,4.015963,4.014976,149.166928,147.803629,8.233838e+07
9,2026 M07,STE,137.337812,-0.500705,6.860290,6.367866,6.367866,6.367866,2.040993,1.411340,0.220843,5.149308,4.390121,4.393802,4.390121,150.515507,149.263936,5.938374e+07



--- Population breakdown (first 5 vintage-LOB combos) ---
  STE 2025 M10: 974 total | 951 bb>0 | 884 scored | AF total=32,707,283
  STE 2025 M11: 1,531 total | 1,504 bb>0 | 1,403 scored | AF total=48,131,535
  STE 2025 M12: 1,600 total | 1,586 bb>0 | 1,521 scored | AF total=49,068,667
  STE 2026 M01: 1,379 total | 1,374 bb>0 | 1,327 scored | AF total=42,066,006
  STE 2026 M02: 1,790 total | 1,783 bb>0 | 1,712 scored | AF total=51,895,657


In [20]:
# =============================================================================
# CELL 9: EXCEL EXPORT (AGGREGATES ONLY)
# =============================================================================


AGG_SHEET_MAP = {'q': 'Data Tables (Q)', 'm': 'Data Tables (M)', 'w': 'Data Tables (W)'}
EXCEL_OUTPUT = '../output/ste_ragu_individual.xlsx'

METRIC_ROWS = [
    ('Model Score',                         'model_score'),
    ('Gross Loss Impact',                   'gross_loss_impact'),
    ('Recovery Impact',                     'recovery_impact'),
    ('Recovery Impact (Adj Prop)',          'recovery_impact_adj_prop'),
    ('Recovery Impact (Adj Contrib)',       'recovery_impact_adj_contrib'),
    ('Recovery Impact (Adj Exact)',         'recovery_impact_adj_exact'),
    ('LTV Impact',                          'ltv_impact'),
    ('LTV Impact (Adj Prop)',              'ltv_impact_adj_prop'),
    ('LTV Impact (Adj Contrib)',           'ltv_impact_adj_contrib'),
    ('LTV Impact (Adj Exact)',             'ltv_impact_adj_exact'),
    ('APR Impact',                          'apr_impact'),
    ('RAGU Score',                          'ragu_score'),
    ('RAGU Score (Agg)',                    'ragu_score_agg'),
    ('Amount Financed',                     'amt_financed'),
    ('Weighted LTV',                        'ltv'),
    ('Weighted APR',                        'apr'),
]

agg_sheet = AGG_SHEET_MAP[granularity]

if os.path.exists(EXCEL_OUTPUT):
    wb = openpyxl.load_workbook(EXCEL_OUTPUT)
    if agg_sheet in wb.sheetnames:
        del wb[agg_sheet]
    ws_agg = wb.create_sheet(agg_sheet)
else:
    wb = openpyxl.Workbook()
    ws_agg = wb.active
    ws_agg.title = agg_sheet

sorted_vintages = sorted(vintage_lob_df['vintage'].unique())
all_export_lobs = LOBS + list(ROLLUP_GROUPS.keys())
current_row = 1

for lob in all_export_lobs:
    lob_data = vintage_lob_df[vintage_lob_df.lob == lob].set_index('vintage')

    ws_agg.cell(row=current_row, column=1, value=lob)
    for col_idx, v in enumerate(sorted_vintages, start=2):
        ws_agg.cell(row=current_row, column=col_idx, value=v)
    current_row += 1

    for label, col_key in METRIC_ROWS:
        ws_agg.cell(row=current_row, column=1, value=label)
        for col_idx, v in enumerate(sorted_vintages, start=2):
            if v in lob_data.index:
                ws_agg.cell(row=current_row, column=col_idx, value=lob_data.loc[v, col_key])
        current_row += 1

    current_row += 1

wb.save(EXCEL_OUTPUT)
print(f"Aggregated sheet '{agg_sheet}': {len(all_export_lobs)} groups x {len(sorted_vintages)} periods")
print(f"Saved to {EXCEL_OUTPUT}")

Aggregated sheet 'Data Tables (M)': 1 groups x 11 periods
Saved to ../output/ste_ragu_individual.xlsx


In [21]:
# =============================================================================
# CELL 10: SANDBOX EXPORT -- RAGU INPUTS (ACCOUNT-LEVEL)
# =============================================================================

TABLE = 'sandbox.gl_ragu_individual_ste'
BATCH_SIZE = 1000

def _fmt(v, dp=4):
    return 'NULL' if pd.isna(v) else f'{v:.{dp}f}'

upload_df = acct_df[[
    'account_number', 'lob', 'book_date', 'model_score', 'gross_loss_impact',
    'recovery_multiplier', 'ltv', 'apr', 'amt_financed', 'ragu_score'
]].copy()
upload_df['gross_loss_ragu'] = upload_df['model_score'] + upload_df['gross_loss_impact']
upload_df = upload_df.reset_index(drop=True)

print(f"Rows to upload: {len(upload_df):,}")
display(upload_df.head(10))

if granularity == 'blahbloobleeeh':
    print(f"WARNING: Sandbox upload skipped (granularity='{granularity}'). "
          "Only runs for weekly ('w') granularity.")
else:
    with pyodbc.connect("DSN=Redshift_prod_new") as conn:
        cur = conn.cursor()
        cur.execute(f"DROP TABLE IF EXISTS {TABLE};")
        cur.execute(f"""
            CREATE TABLE {TABLE} (
                account_number      BIGINT,
                lob                 VARCHAR(25),
                book_date           DATE,
                model_score         FLOAT,
                gross_loss_impact   FLOAT,
                gross_loss_ragu     FLOAT,
                ragu_score          FLOAT,
                recovery_multiplier FLOAT,
                ltv                 FLOAT,
                apr                 FLOAT,
                amt_financed        FLOAT
            );
        """)
        conn.commit()

        for start in range(0, len(upload_df), BATCH_SIZE):
            batch = upload_df.iloc[start:start + BATCH_SIZE]
            values = ", ".join(
                f"({int(r.account_number)}, '{r.lob}', '{r.book_date.strftime('%Y-%m-%d')}', "
                f"{_fmt(r.model_score)}, {_fmt(r.gross_loss_impact)}, {_fmt(r.gross_loss_ragu)}, "
                f"{_fmt(r.ragu_score)}, "
                f"{_fmt(r.recovery_multiplier)}, {_fmt(r.ltv)}, {_fmt(r.apr)}, {_fmt(r.amt_financed, 2)})"
                for r in batch.itertuples(index=False)
            )
            cur.execute(f"INSERT INTO {TABLE} VALUES {values}")
            conn.commit()

        print(f"Uploaded {len(upload_df):,} rows to {TABLE}")

        cur.execute("CALL sandbox.util_table_grant('gl_ragu_individual_ste')")
        conn.commit()
        print(f"Granted access on {TABLE}")

Rows to upload: 20,741


,account_number,lob,book_date,model_score,gross_loss_impact,recovery_multiplier,ltv,apr,amt_financed,ragu_score,gross_loss_ragu
0,90125158207,STE,2026-01-14,122.0,-0.236750,0.534287,1.581435,0.2700,28110.00,122.891299,121.763250
1,90125118651,STE,2025-11-13,134.0,-5.836232,0.620002,1.448036,0.1800,34101.25,143.333417,128.163768
2,90125297562,STE,2026-06-25,130.0,2.057500,0.495015,1.742287,0.2639,25393.84,129.376321,132.057500
3,90125280684,STE,2026-06-02,141.0,-5.646665,0.575841,1.170246,0.2239,20420.79,149.680693,135.353335
4,90125291269,STE,2026-06-16,137.0,2.745775,0.514770,1.635020,0.2669,20683.00,139.006759,139.745775
5,90125142444,STE,2025-12-22,132.0,2.332810,0.563568,1.518376,0.2060,26154.02,142.579748,134.332810
6,90125265330,STE,2026-05-14,132.0,-4.073135,0.494664,1.720444,0.2548,17419.50,126.219348,127.926865
7,90125268594,STE,2026-05-19,135.0,-3.543463,0.511090,1.440931,0.2718,20533.27,133.049643,131.456537
8,90125208554,STE,2026-03-12,132.0,-6.215366,0.519945,1.485696,0.2500,18942.63,128.807553,125.784634
9,90125135975,STE,2025-12-09,137.0,1.105990,0.495595,1.289948,0.2698,21509.88,141.514175,138.105990


Uploaded 20,741 rows to sandbox.gl_ragu_individual_ste
Granted access on sandbox.gl_ragu_individual_ste


In [22]:
# =============================================================================
# CELL 11: DIAGNOSTICS
# =============================================================================

print("=== STE Individual RAGU Score Summary ===")
print(f"Accounts: {len(acct_df):,}")
print(f"Vintages: {acct_df.vintage.nunique()}")
print(f"Period range: {acct_df.vintage.min()} to {acct_df.vintage.max()}")

scored = acct_df[acct_df.ragu_score.notna()]
print(f"\nScored accounts: {len(scored):,}")
print(f"Mean RAGU Score: {scored.ragu_score.mean():.2f}")
print(f"Std RAGU Score:  {scored.ragu_score.std():.2f}")
print(f"Min RAGU Score:  {scored.ragu_score.min():.2f}")
print(f"Max RAGU Score:  {scored.ragu_score.max():.2f}")

print("\n=== Component Averages (scored population) ===")
for col in ['model_score', 'gross_loss_impact', 'recovery_impact', 'ltv_impact', 'apr_impact']:
    if col in scored.columns:
        print(f"  {col}: {scored[col].mean():.4f}")

print("\n=== Aggregate vs Individual Comparison ===")
print(f"Aggregate RAGU (ragu_score_agg) mean: {scored.ragu_score_agg.mean():.4f}")
print(f"Individual RAGU (ragu_score) mean:    {scored.ragu_score.mean():.4f}")
print(f"Difference:                           {(scored.ragu_score_agg.mean() - scored.ragu_score.mean()):.4f}")

print("\n=== SFS Override Coverage ===")
print(f"Vintage-LOB aggregation rows: {len(vintage_lob_df)}")
display(vintage_lob_df[['vintage', 'lob', 'model_score', 'ragu_score', 'ragu_score_agg', 'amt_financed']].head(10))

=== STE Individual RAGU Score Summary ===
Accounts: 20,741
Vintages: 11
Period range: 2025 M10 to 2026 M08

Scored accounts: 19,417
Mean RAGU Score: 142.77
Std RAGU Score:  12.48
Min RAGU Score:  103.91
Max RAGU Score:  200.11

=== Component Averages (scored population) ===
  model_score: 134.2537
  gross_loss_impact: -0.3706
  recovery_impact: 2.0448
  ltv_impact: 5.7514
  apr_impact: 1.0925

=== Aggregate vs Individual Comparison ===
Aggregate RAGU (ragu_score_agg) mean: 141.8654
Individual RAGU (ragu_score) mean:    142.7718
Difference:                           -0.9064

=== SFS Override Coverage ===
Vintage-LOB aggregation rows: 11


,vintage,lob,model_score,ragu_score,ragu_score_agg,amt_financed
0,2025 M10,STE,131.021335,140.090855,138.846145,3.270728e+07
1,2025 M11,STE,131.365156,139.653353,138.394426,4.813154e+07
2,2025 M12,STE,131.651392,139.242640,138.000396,4.906867e+07
3,2026 M01,STE,132.248966,140.714295,139.491130,4.206601e+07
4,2026 M02,STE,133.111325,141.372814,140.169387,5.189566e+07
5,2026 M03,STE,134.396978,143.259055,142.167582,1.078950e+08
6,2026 M04,STE,135.069052,145.714137,144.468842,9.062796e+07
7,2026 M05,STE,136.238790,148.585508,147.285660,7.272996e+07
8,2026 M06,STE,136.526672,149.166928,147.803629,8.233838e+07
9,2026 M07,STE,137.337812,150.515507,149.263936,5.938374e+07
